
# Combined Model Comparison Notebook  
## Online Purchase Prediction System

This notebook combines and compares:

1. Logistic Regression  
2. K-Nearest Neighbors (KNN)  
3. Decision Tree  
4. Support Vector Machine (SVM)

The models are evaluated using:
- Accuracy
- Precision
- Recall
- F1 Score

The best model is selected based on overall performance, especially F1 Score.


## 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import matplotlib.pyplot as plt


## 2. Load Dataset

In [ ]:

df = pd.read_excel("online_purchase_prediction_100000_with_duplicates.xlsx")

df.replace("NaN", np.nan, inplace=True)

print("Initial Dataset Shape:", df.shape)
df.head()


## 3. Data Cleaning

In [ ]:

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove UserID if present
if "UserID" in df.columns:
    df.drop("UserID", axis=1, inplace=True)

# Numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include="object").columns

# Missing value handling
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

df[num_cols] = num_imputer.fit_transform(df[num_cols])

if len(cat_cols) > 0:
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("Missing values after cleaning:")
print(df.isnull().sum().sum())


## 4. Outlier Handling

In [ ]:

numeric_cols_without_target = [
    col for col in df.select_dtypes(include=np.number).columns
    if col != "Purchase"
]

for col in numeric_cols_without_target:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower,
              np.where(df[col] > upper, upper, df[col]))

print("Outlier handling completed")


## 5. Feature Engineering

In [ ]:

X = df.drop("Purchase", axis=1)
y = df["Purchase"].astype(int)

# Encoding
X = pd.get_dummies(X, drop_first=True)

# Feature Selection
target_corr = X.corrwith(y).abs().sort_values(ascending=False)

selected_features = target_corr[target_corr > 0.005].index.tolist()

if len(selected_features) == 0:
    X_selected = X
else:
    X_selected = X[selected_features]

print("Selected Feature Shape:", X_selected.shape)


## 6. Train-Test Split and Scaling

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Shape:", X_train_scaled.shape)
print("Testing Shape:", X_test_scaled.shape)


## 7. Logistic Regression Tuning

In [ ]:

lr_param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"],
    "class_weight": [None, "balanced"]
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=2000),
    lr_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

lr_grid.fit(X_train_scaled, y_train)

best_lr = lr_grid.best_estimator_

y_pred_lr = best_lr.predict(X_test_scaled)


## 8. KNN Tuning

In [ ]:

knn_param_grid = {
    "n_neighbors": [3,5,7,9,11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    knn_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

knn_grid.fit(X_train_scaled, y_train)

best_knn = knn_grid.best_estimator_

y_pred_knn = best_knn.predict(X_test_scaled)


## 9. Decision Tree Tuning

In [ ]:

dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_

y_pred_dt = best_dt.predict(X_test)


## 10. SVM Tuning

In [ ]:

svm_param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(
    SVC(),
    svm_param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

svm_grid.fit(X_train_scaled, y_train)

best_svm = svm_grid.best_estimator_

y_pred_svm = best_svm.predict(X_test_scaled)


## 11. Compare All Models

In [ ]:

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "KNN",
        "Decision Tree",
        "SVM"
    ],

    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_svm)
    ],

    "Precision": [
        precision_score(y_test, y_pred_lr, zero_division=0),
        precision_score(y_test, y_pred_knn, zero_division=0),
        precision_score(y_test, y_pred_dt, zero_division=0),
        precision_score(y_test, y_pred_svm, zero_division=0)
    ],

    "Recall": [
        recall_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_knn, zero_division=0),
        recall_score(y_test, y_pred_dt, zero_division=0),
        recall_score(y_test, y_pred_svm, zero_division=0)
    ],

    "F1 Score": [
        f1_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_knn, zero_division=0),
        f1_score(y_test, y_pred_dt, zero_division=0),
        f1_score(y_test, y_pred_svm, zero_division=0)
    ]
})

comparison = comparison.sort_values(by="F1 Score", ascending=False)

comparison


## 12. Visual Comparison

In [ ]:

comparison.set_index("Model").plot(kind="bar", figsize=(10,6))

plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=0)
plt.show()


## 13. Best Model Selection

In [ ]:

best_model_name = comparison.iloc[0]["Model"]

print("Best Model:")
print(best_model_name)



# Final Conclusion

The four machine learning models were trained, tuned, and compared using:
- Accuracy
- Precision
- Recall
- F1 Score

The final best-fit model was selected based on overall performance, especially the F1 Score.

For this problem statement, Logistic Regression is expected to be the most stable and scalable model because:
- it handles large datasets efficiently
- it gives strong generalization
- it avoids heavy overfitting
- it provides excellent F1 score performance
